# Example 12.5 / 3: Compare with the Extensive Form

Companion notebook for **12.5 Decomposition** in the AMPL lab Documentation, section "Compare with the Extensive Form". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Compare with the Extensive Form

On a small instance, solve the extensive form (every scenario's recourse in one model) and
compare it with the Benders result. The table lists this and the other safeguards on the
two-scenario instance:

| Safeguard | Concrete example on the two-scenario instance | Result |
|---|---|---|
| bounded first stage and seeded master | `Build <= 10` and `theta >= 0` | the first master solves with $x=0$ and lower bound 0; with `var theta;` free and no cut yet, nothing bounds `theta` below, so the first master is unbounded |
| iteration log | the `Build`, `LB`, `UB` and cut lines of the Benders run | three iterations, gap 18, 3, then 0 |
| add only violated cuts | the cut at $x=8$, $\theta\ge0+0x$, with master $\theta=0$ | violation 0, so the loop stops without adding it |
| extensive-form comparison | the model below | $x=8$ and cost 8, as in the Benders run |

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set SCENARIOS;                   # demand scenarios
param probability {SCENARIOS} >= 0;     # scenario probability
param demand {SCENARIOS} >= 0;   # units demanded per scenario
param build_cost >= 0;           # cost per unit of capacity
param max_build >= 0;            # capacity limit, units
param outsource_cost >= 0;       # cost per outsourced unit

var Build >= 0, <= max_build;    # first stage: capacity built
var Produce {SCENARIOS} >= 0;    # second stage: in-house units per scenario
var Outsource {SCENARIOS} >= 0;  # second stage: outsourced units per scenario

minimize ExpectedCost:           # build cost plus probability-weighted recourse
    build_cost * Build + sum {s in SCENARIOS} probability[s] * outsource_cost * Outsource[s];

# meet each scenario demand
subject to Supply {s in SCENARIOS}: Produce[s] + Outsource[s] >= demand[s];
# production within the built capacity
subject to CapacityUse {s in SCENARIOS}: Produce[s] <= Build;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["SCENARIOS"] = ["low", "high"]
ampl.param["probability"] = {"low": 0.5, "high": 0.5}
ampl.param["demand"] = {"low": 4, "high": 8}
ampl.param["build_cost"] = 1
ampl.param["max_build"] = 10
ampl.param["outsource_cost"] = 3

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, "Build", ampl.var["Build"].value(),
      "ExpectedCost", ampl.obj["ExpectedCost"].value())

solved Build 8.0 ExpectedCost 8.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set SCENARIOS := low high;
param: probability demand :=
    low    0.5   4
    high   0.5   8 ;
param build_cost := 1;
param max_build := 10;
param outsource_cost := 3;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0;
solve;
display solve_result, Build, ExpectedCost;

solve_result = solved
Build = 8
ExpectedCost = 8

